In [9]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [10]:
# Task 1

print("Shape:", df.shape, "\n")


Shape: (7043, 21) 



In [11]:

print("Dtypes:\n", df.dtypes, "\n")


Dtypes:
 customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object 



In [12]:

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Missing values (NaN):\n", missing, "\n")


Missing values (NaN):
 Series([], dtype: int64) 



In [13]:

temp = pd.to_numeric(df['TotalCharges'], errors='coerce')
problem_rows = df[temp.isna()]
print("Baris bermasalah TotalCharges:", len(problem_rows))
print(problem_rows[['customerID', 'TotalCharges']].head(), "\n")

print("Unique SeniorCitizen:", df['SeniorCitizen'].unique(), "\n")

print("Ringkasan Audit:")
print("- TotalCharges: string kosong/spasi (~{} baris)".format(len(problem_rows)))
print("- SeniorCitizen: valid (0 dan 1 saja)\n")



Baris bermasalah TotalCharges: 11
      customerID TotalCharges
488   4472-LVYGI             
753   3115-CZMZD             
936   5709-LVOEQ             
1082  4367-NUYAO             
1340  1371-DWPAZ              

Unique SeniorCitizen: [0 1] 

Ringkasan Audit:
- TotalCharges: string kosong/spasi (~11 baris)
- SeniorCitizen: valid (0 dan 1 saja)



In [ ]:

object_cols = df.select_dtypes(include=['object', 'string']).columns

for col in object_cols:
    print(f"\n{col}")
    print(df[col].unique())

print("\nInsight:")
print("- 'No internet service' dan 'No phone service' = kategori valid (bukan missing)\n")

# tenure = 0
tenure = df[df['tenure'] == 0]
print("Jumlah tenure = 0:", len(tenure), "\n")




customerID
<StringArray>
['7590-VHVEG', '5575-GNVDE', '3668-QPYBK', '7795-CFOCW', '9237-HQITU',
 '9305-CDSKC', '1452-KIOVK', '6713-OKOMC', '7892-POOKP', '6388-TABGU',
 ...
 '9767-FFLEM', '0639-TSIQW', '8456-QDAVC', '7750-EYXWZ', '2569-WGERO',
 '6840-RESVB', '2234-XADUH', '4801-JZAZL', '8361-LTMKD', '3186-AJIEK']
Length: 7043, dtype: str

gender
<StringArray>
['Female', 'Male']
Length: 2, dtype: str

Partner
<StringArray>
['Yes', 'No']
Length: 2, dtype: str

Dependents
<StringArray>
['No', 'Yes']
Length: 2, dtype: str

PhoneService
<StringArray>
['No', 'Yes']
Length: 2, dtype: str

MultipleLines
<StringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str

InternetService
<StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str

OnlineSecurity
<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

OnlineBackup
<StringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str

DeviceProtection
<StringArray>
['No', 'Yes', 'No internet service']


In [15]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

missing_totalcharges = df['TotalCharges'].isna().sum()

median_tc = df['TotalCharges'].median()

df['TotalCharges'] = df['TotalCharges'].fillna(median_tc)

print("TotalCharges missing diisi median:", median_tc)

tenure_zero_count = df[df['tenure'] == 0].shape[0]

df.loc[df['tenure'] == 0, 'tenure'] = np.nan

df['tenure'] = df.groupby(
    ['InternetService', 'PhoneService']
)['tenure'].transform(lambda x: x.fillna(x.median()))

print("\nMissing setelah cleaning:")
print(df[['TotalCharges', 'tenure']].isnull().sum(), "\n")

TotalCharges missing diisi median: 1397.475

Missing setelah cleaning:
TotalCharges    0
tenure          0
dtype: int64 



In [16]:
print("Kolom          | Masalah Ditemukan              | Penanganan               | Baris Terdampak\n")
print(f"TotalCharges   | String kosong/spasi            | Konversi + isi median    | {missing_totalcharges} baris")
print(f"tenure         | Nilai 0 tidak valid            | Ganti NaN + isi groupby  | {tenure_zero_count} baris")


print("\nAlasan penggunaan median:")
print("- Median lebih tahan terhadap outlier dibanding mean pada distribusi skewed seperti TotalCharges.")


Kolom          | Masalah Ditemukan              | Penanganan               | Baris Terdampak

TotalCharges   | String kosong/spasi            | Konversi + isi median    | 11 baris
tenure         | Nilai 0 tidak valid            | Ganti NaN + isi groupby  | 11 baris

Alasan penggunaan median:
- Median lebih tahan terhadap outlier dibanding mean pada distribusi skewed seperti TotalCharges.
